# Module 03: Feature Engineering & Preprocessing Pipeline
## Build a Robust, Leak-Free Machine Learning Preprocessing Pipeline

This notebook demonstrates end-to-end, leak-free feature engineering and preprocessing for tabular data (Credit Default Risk evaluation):
1. **Dataset Creation & Ingestion:** Loading a realistic multi-attribute tabular dataset (numerical, nominal, ordinal features with realistic missingness).
2. **Strict Train/Test Split:** Partitioning the dataset before any imputation, scaling, or encoding to eliminate target and feature leakage.
3. **Modular Scikit-Learn Pipeline & ColumnTransformer:** Integrating median/iterative numerical imputation, robust/standard scaling, categorical missing value handling, and One-Hot / Target Encoding.
4. **Feature Importance & Correlation Analysis:** Correlation analysis, Mutual Information classification scoring, and Random Forest feature ranking.
5. **Reusable Model Fitting & Evaluation:** End-to-end classification pipeline evaluation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, RobustScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

# Set seed for absolute reproducibility
np.random.seed(42)

## 1. Load Complex Tabular Dataset (Credit Default Risk)
We construct a realistic 1,000-record Credit Default Risk dataset containing:
- **Numerical features:** `age`, `annual_income`, `debt_to_income_ratio`, `credit_score`, `open_credit_lines`
- **Categorical features:** `home_ownership` (Rent, Own, Mortgage), `loan_purpose` (Debt Consolidation, Home Improvement, Business, Education, Personal), `employment_type` (Salaried, Self-Employed, Freelance, Unemployed)
- **Realistic missingness & noise:** Representing missing values in income, employment type, and credit score.

In [ ]:
n_samples = 1200

age = np.random.randint(21, 68, size=n_samples)
annual_income = np.random.exponential(scale=600000, size=n_samples) + 200000
credit_score = np.clip(np.random.normal(loc=680, scale=80, size=n_samples), 350, 850)
debt_to_income_ratio = np.random.uniform(0.05, 0.65, size=n_samples)
open_credit_lines = np.random.poisson(lam=4, size=n_samples) + 1

home_ownership = np.random.choice(['RENT', 'OWN', 'MORTGAGE'], size=n_samples, p=[0.45, 0.20, 0.35])
loan_purpose = np.random.choice(
    ['DEBT_CONSOLIDATION', 'HOME_IMPROVEMENT', 'BUSINESS', 'EDUCATION', 'PERSONAL'],
    size=n_samples, p=[0.40, 0.20, 0.15, 0.15, 0.10]
)
employment_type = np.random.choice(
    ['SALARIED', 'SELF_EMPLOYED', 'FREELANCE', 'UNEMPLOYED'],
    size=n_samples, p=[0.55, 0.25, 0.15, 0.05]
)

# Latent default probability formula
latent_risk = (
    - 0.003 * credit_score
    + 4.5 * debt_to_income_ratio
    - 0.0000015 * annual_income
    + (home_ownership == 'RENT') * 0.45
    + (employment_type == 'UNEMPLOYED') * 1.20
    + (loan_purpose == 'BUSINESS') * 0.35
)
prob_default = 1 / (1 + np.exp(-(latent_risk - np.median(latent_risk))))
default_label = (np.random.uniform(0, 1, size=n_samples) < prob_default).astype(int)

df = pd.DataFrame({
    'age': age,
    'annual_income': annual_income,
    'credit_score': credit_score,
    'debt_to_income_ratio': debt_to_income_ratio,
    'open_credit_lines': open_credit_lines,
    'home_ownership': home_ownership,
    'loan_purpose': loan_purpose,
    'employment_type': employment_type,
    'default_risk': default_label
})

# Inject realistic missingness
df.loc[df.sample(frac=0.06, random_state=101).index, 'annual_income'] = np.nan
df.loc[df.sample(frac=0.04, random_state=102).index, 'credit_score'] = np.nan
df.loc[df.sample(frac=0.05, random_state=103).index, 'employment_type'] = np.nan

print("Dataset Shape:", df.shape)
print("\nMissing Values Summary:")
print(df.isnull().sum())
print("\nClass Distribution:")
print(df['default_risk'].value_counts(normalize=True))

## 2. Train / Test Split Before Transformation (Zero Leakage Guarantee)
> **Data Leakage Safeguard:** Fitting scalers, encoders, or imputers on the entire dataset leaks information from the evaluation fold into training distributions. Splitting *prior* to fitting ensures evaluation integrity.

In [ ]:
X = df.drop(columns=['default_risk'])
y = df['default_risk']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training set dimensions: {X_train.shape}")
print(f"Test set dimensions:     {X_test.shape}")

## 3. Implement Reusable Scikit-Learn ColumnTransformer & Pipelines
- **Numerical Pipeline:** Median imputation for robustness against skew, followed by `StandardScaler` / `RobustScaler`.
- **Categorical Pipeline:** Constant most frequent imputation (`'missing'` marker), followed by `OneHotEncoder(handle_unknown='ignore', sparse_output=False)`.

In [ ]:
numeric_features = ['age', 'annual_income', 'credit_score', 'debt_to_income_ratio', 'open_credit_lines']
categorical_features = ['home_ownership', 'loan_purpose', 'employment_type']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='MISSING')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# Fit preprocessor strictly on training data
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

# Extract transformed feature names for inspection
cat_encoded_names = preprocessor.named_transformers_['cat'].named_steps['encoder'].get_feature_names_out(categorical_features)
all_feature_names = numeric_features + list(cat_encoded_names)

print(f"Transformed feature count: {len(all_feature_names)}")
print("Sample engineered feature names:", all_feature_names[:8])

## 4. Correlation & Feature Importance Analysis
Conducting:
1. **Mutual Information Classification (Non-linear relevance)**
2. **Tree-Based Feature Importance (Random Forest Gini Importance)**

In [ ]:
# Compute Mutual Information Scores
mi_scores = mutual_info_classif(X_train_transformed, y_train, random_state=42)
mi_df = pd.DataFrame({'Feature': all_feature_names, 'Mutual_Information': mi_scores})
mi_df = mi_df.sort_values(by='Mutual_Information', ascending=False).reset_index(drop=True)

# Fit Random Forest on preprocessed training data for Gini Importance
rf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf.fit(X_train_transformed, y_train)

rf_df = pd.DataFrame({'Feature': all_feature_names, 'RF_Importance': rf.feature_importances_})
rf_df = rf_df.sort_values(by='RF_Importance', ascending=False).reset_index(drop=True)

# Merge rankings
importance_summary = pd.merge(mi_df, rf_df, on='Feature')
print("=== Top 10 Feature Importance Table ===")
print(importance_summary.head(10).to_string(index=False))

# Plot top feature rankings
plt.figure(figsize=(10, 6))
top_10 = importance_summary.head(10)
sns.barplot(data=top_10, x='RF_Importance', y='Feature', palette='Blues_r')
plt.title('Top 10 Feature Importance (Random Forest)', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

## 5. End-to-End Reusable ML Pipeline & Evaluation
We assemble the preprocessor and estimator into a unified Scikit-Learn `Pipeline`.

In [ ]:
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=150, max_depth=5, random_state=42))
])

# Train entire pipeline end-to-end
full_pipeline.fit(X_train, y_train)

# Predict on unseen test data
y_pred = full_pipeline.predict(X_test)
y_prob = full_pipeline.predict_proba(X_test)[:, 1]

print("=== Out-of-Sample Classification Report ===")
print(classification_report(y_test, y_pred, target_names=['Non-Default', 'Default']))
print(f"Test ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")